# 🔧 Дескрипторы в Python: управление доступом к атрибутам

<div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 20px; border-radius: 10px; color: white;">
  <h2 style="color: white;">Как работает <code>@property</code> и не только</h2>
  <p>Дескрипторы — это мощный механизм Python для управления доступом к атрибутам. Они лежат в основе <code>@property</code>, методов класса, статических методов и многих других магических возможностей языка.</p>
  <p>Сегодня мы разберём протокол дескрипторов и научимся создавать свои дескрипторы для валидации, ленивой загрузки и многого другого.</p>
</div>

## 🎯 Цели лекции

- Понять, что такое дескрипторы и для чего они нужны
- Изучить протокол дескрипторов: `__get__`, `__set__`, `__delete__`
- Различать data-дескрипторы и non-data-дескрипторы
- Увидеть, как реализован `@property` под капотом
- Научиться создавать свои дескрипторы для валидации, типизации, ленивых вычислений
- Понять порядок поиска атрибутов в Python

## 📚 План лекции

1. **Что такое дескриптор?** — понятие, зачем нужен
2. **Протокол дескриптора** — `__get__`, `__set__`, `__delete__`
3. **Data vs Non-data дескрипторы** — в чём разница и как влияет на поиск
4. **`@property` под капотом** — как он реализован с помощью дескриптора
5. **Создание своих дескрипторов** — практические примеры
   - Валидация (положительное число, длина строки)
   - Типобезопасность (атрибут определённого типа)
   - Ленивая инициализация (вычисление при первом доступе)
   - Дескриптор-логгер (отслеживание доступа)
6. **Порядок поиска атрибутов** — как Python ищет атрибуты и где участвуют дескрипторы
7. **Частые ошибки и подводные камни**
8. **Итоги и практические советы**

In [ ]:
# Начнём с простого примера: обычный класс без дескрипторов
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

p = Person("Анна", 25)
p.age = -5  # Проблема: отрицательный возраст, но класс этого не контролирует
print(p.age)  # -5

## 1. Что такое дескриптор?

**Дескриптор** — это атрибут класса, который реализует протокол из одного или нескольких методов: `__get__`, `__set__`, `__delete__`. Дескриптор позволяет переопределить доступ к атрибуту экземпляра.

Простыми словами: дескриптор — это «посредник», который управляет тем, как читается, записывается или удаляется атрибут объекта.

### Зачем нужны дескрипторы?
- **Валидация** — проверять значения перед присваиванием (например, возраст не может быть отрицательным)
- **Преобразование типов** — автоматически приводить значение к нужному типу
- **Ленивая загрузка** — вычислять значение только при первом обращении
- **Логирование** — отслеживать доступ к атрибутам
- **Создание управляемых атрибутов** — аналогично `@property`, но переиспользуемых

## 2. Протокол дескриптора

Протокол состоит из трёх методов:

```python
descr.__get__(self, obj, type=None) -> value
descr.__set__(self, obj, value) -> None
descr.__delete__(self, obj) -> None

- self — сам дескриптор (экземпляр класса-дескриптора)
- obj — экземпляр класса, у которого происходит доступ к атрибуту (может быть None, если доступ через класс)
- type — класс объекта obj (обычно совпадает с type(obj)) 

Если дескриптор реализует только __get__ — это non-data дескриптор.
Если реализует __set__ или __delete__ — это data-дескриптор. Разница важна для порядка поиска атрибутов.

In [ ]:
# Пример простейшего дескриптора
class Ten:
    """Всегда возвращает 10, игнорируя попытки записи."""
    def __get__(self, obj, objtype=None):
        return 10

    def __set__(self, obj, value):
        print(f"Попытка установить {value}, но мы игнорируем")

class MyClass:
    x = Ten()  # x — дескриптор

obj = MyClass()
print(obj.x)  # 10
obj.x = 42    # Попытка установить 42, но мы игнорируем
print(obj.x)  # 10

In [ ]:
# Пример дескриптора с хранением значений
class PositiveNumber:
    """Дескриптор для положительного числа."""
    def __init__(self, name):
        self.name = name  # имя атрибута для хранения в __dict__

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name, None)

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} должен быть положительным")
        obj.__dict__[self.name] = value

    def __delete__(self, obj):
        del obj.__dict__[self.name]

class Person:
    age = PositiveNumber("age")
    score = PositiveNumber("score")

    def __init__(self, name, age, score):
        self.name = name
        self.age = age
        self.score = score

try:
    p = Person("Иван", -5, 100)
except ValueError as e:
    print(f"Ошибка: {e}")

p = Person("Мария", 30, 95)
print(p.age, p.score)
# p.age = -10  # ValueError

## 3. Data vs Non-data дескрипторы

- **Data-дескриптор** — реализует `__set__` и/или `__delete__`. Он «перехватывает» запись и удаление атрибута, а также чтение (если есть `__get__`).
- **Non-data дескриптор** — реализует только `__get__`. Он «перехватывает» только чтение, но не запись.

Это различие важно для **порядка поиска атрибутов** (о котором позже). Data-дескрипторы имеют более высокий приоритет, чем атрибуты экземпляра.

In [ ]:
# Демонстрация разницы между data и non-data дескрипторами
class NonDataDescriptor:
    def __get__(self, obj, objtype=None):
        return "non-data value"

class DataDescriptor:
    def __get__(self, obj, objtype=None):
        return "data value"
    def __set__(self, obj, value):
        pass  # просто перехватываем установку

class Test:
    non_data = NonDataDescriptor()
    data = DataDescriptor()

t = Test()
print("--- Non-data descriptor ---")
print(t.non_data)      # "non-data value"
t.non_data = "override"
print(t.non_data)      # "override" (атрибут экземпляра переопределил дескриптор)
del t.non_data
print(t.non_data)      # "non-data value" (вернулся дескриптор)

print("--- Data descriptor ---")
print(t.data)          # "data value"
t.data = "override"
print(t.data)          # "data value" (data-дескриптор не дал переопределить)
# атрибут экземпляра t.__dict__['data'] существует, но не используется
print(t.__dict__['data'])  # "override" (но t.data игнорирует его)

## 4. `@property` под капотом

Декоратор `@property` — это удобная обёртка для создания data-дескриптора. По сути, `property` — это класс, реализующий протокол дескриптора.

Когда мы пишем:
```python
class Circle:
    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value <= 0:
            raise ValueError("Радиус должен быть положительным")
        self._radius = value

Python создаёт экземпляр класса property с методами ```fget```, ```fset```, `fdel`. Этот экземпляр является data-дескриптором.

Давайте реализуем упрощённый аналог `property`:

In [ ]:


class MyProperty:
    """Упрощённая реализация декоратора property."""
    def __init__(self, fget=None, fset=None, fdel=None):
        self.fget = fget
        self.fset = fset
        self.fdel = fdel

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.fget is None:
            raise AttributeError("unreadable attribute")
        return self.fget(obj)

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel(obj)

    def setter(self, fset):
        self.fset = fset
        return self

    def deleter(self, fdel):
        self.fdel = fdel
        return self

# Использование своего property
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @MyProperty
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value <= 0:
            raise ValueError("Radius must be positive")
        self._radius = value

c = Circle(5)
print(c.radius)
c.radius = 10
print(c.radius)
# c.radius = -1  # ValueError

## 5. Создание своих дескрипторов: практические примеры

### 5.1 Дескриптор валидации (тип и диапазон)

In [ ]:
class Typed:
    """Дескриптор для проверки типа."""
    def __init__(self, expected_type, name=None):
        self.expected_type = expected_type
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name, None)

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name} должен быть {self.expected_type.__name__}, получен {type(value).__name__}")
        obj.__dict__[self.name] = value

class PositiveInt:
    """Дескриптор для положительного целого числа."""
    def __init__(self, name=None):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name, None)

    def __set__(self, obj, value):
        if not isinstance(value, int):
            raise TypeError(f"{self.name} должен быть int")
        if value <= 0:
            raise ValueError(f"{self.name} должен быть положительным")
        obj.__dict__[self.name] = value

class Employee:
    name = Typed(str, "name")
    age = PositiveInt("age")
    salary = Typed(float, "salary")

    def __init__(self, name, age, salary):
        self.name = name
        self.age = age
        self.salary = salary

e = Employee("Иван", 30, 50000.0)
print(e.name, e.age, e.salary)
try:
    e2 = Employee("Петр", -5, 60000)
except ValueError as e:
    print(f"Ошибка: {e}")
try:
    e3 = Employee("Сидор", 25, "много")
except TypeError as e:
    print(f"Ошибка: {e}")

### 5.2 Дескриптор ленивой инициализации (ленивое свойство)

In [ ]:
class LazyProperty:
    """Дескриптор, который вычисляет значение один раз при первом доступе."""
    def __init__(self, function):
        self.function = function
        self.name = function.__name__

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        value = self.function(obj)
        # Заменяем дескриптор на обычный атрибут (кэшируем)
        obj.__dict__[self.name] = value
        return value

class DataProcessor:
    def __init__(self, data):
        self.data = data

    @LazyProperty
    def processed_data(self):
        print("Вычисляем processed_data (дорогая операция)...")
        # Симулируем долгую обработку
        import time
        time.sleep(1)
        return [x * 2 for x in self.data]

dp = DataProcessor([1, 2, 3, 4, 5])
print("Первый доступ:")
print(dp.processed_data)  # вычисляется
print("Второй доступ:")
print(dp.processed_data)  # берётся из __dict__

### 5.3 Дескриптор-логгер

In [ ]:
class LoggedAccess:
    """Дескриптор, логирующий доступ к атрибуту."""
    def __init__(self, name=None):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        print(f"[LOG] Чтение {self.name} = {obj.__dict__.get(self.name)}")
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        print(f"[LOG] Запись {self.name} = {value}")
        obj.__dict__[self.name] = value

    def __delete__(self, obj):
        print(f"[LOG] Удаление {self.name}")
        del obj.__dict__[self.name]

class User:
    username = LoggedAccess("username")
    email = LoggedAccess("email")

    def __init__(self, username, email):
        self.username = username
        self.email = email

u = User("alice", "alice@example.com")
u.username = "alice_updated"
print(u.username)
del u.username

## 6. Порядок поиска атрибутов в Python (важно!)

Когда вы пишете `obj.attr`, Python ищет атрибут в следующем порядке:

1. **Data-дескрипторы** в классе `obj` и его родительских классах (если есть `__set__` или `__delete__`)
2. **Атрибуты экземпляра** (словарь `obj.__dict__`)
3. **Non-data дескрипторы** в классе и родителях (только `__get__`)
4. **Атрибуты класса** (в `obj.__class__.__dict__`)
5. **Родительские классы** (рекурсивно)
6. **`__getattr__`** (если определён)
7. **`AttributeError`**

Это объясняет, почему data-дескрипторы имеют приоритет над атрибутами экземпляра, а non-data — нет.

In [ ]:
# Демонстрация порядка поиска
class DataDesc:
    def __get__(self, obj, objtype=None):
        return "DataDesc"
    def __set__(self, obj, value):
        pass

class NonDataDesc:
    def __get__(self, obj, objtype=None):
        return "NonDataDesc"

class Demo:
    data = DataDesc()
    non_data = NonDataDesc()

d = Demo()
print("--- Изначально ---")
print(d.data, d.non_data)

print("--- Добавляем атрибуты экземпляра ---")
d.data = "instance data"
d.non_data = "instance non_data"
print(d.data, d.non_data)  # data: всё ещё DataDesc (data-дескриптор блокирует), non_data: "instance non_data"
print("Содержимое __dict__:", d.__dict__)  # {'non_data': 'instance non_data', 'data': 'instance data'}

print("--- Удаляем атрибуты экземпляра ---")
del d.non_data
print(d.non_data)  # снова NonDataDesc
# del d.data  # AttributeError: __delete__ не реализован? но data-дескриптор перехватывает удаление

## 7. Частые ошибки и подводные камни

### 7.1 Забыть про `obj is None` в `__get__`
При доступе через класс (`ClassName.descriptor`) параметр `obj` равен `None`. Нужно это обрабатывать.

### 7.2 Использование одного дескриптора для нескольких атрибутов
Если один дескриптор используется для нескольких атрибутов (например, `age = PositiveNumber()`, `score = PositiveNumber()`), нужно хранить значения в `__dict__` с уникальными ключами. Имя атрибута можно передавать в конструктор.

### 7.3 Data-дескриптор без хранения данных
Если data-дескриптор не хранит значения в `obj.__dict__`, то атрибут станет «незаписываемым» или «одноразовым».

### 7.4 Путаница между дескриптором и атрибутом класса
Дескриптор — это **атрибут класса**, а не экземпляра. Если присвоить дескриптор экземпляру, он перестанет работать как дескриптор (станет обычным атрибутом).

### 7.5 Проблемы с наследованием
Дескрипторы наследуются. Если в дочернем классе переопределить дескриптор, нужно быть внимательным.

In [ ]:
# Пример ошибки: забыли обработать obj is None
class BadDescriptor:
    def __get__(self, obj, objtype=None):
        return obj._value  # ошибка, если obj is None

class MyClass:
    attr = BadDescriptor()

# MyClass.attr  # AttributeError: 'NoneType' object has no attribute '_value'
# Исправление:
class GoodDescriptor:
    def __init__(self, name):
        self.name = name
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

class MyClass2:
    attr = GoodDescriptor("attr")
    def __init__(self, value):
        self.attr = value

print(MyClass2.attr)  # <__main__.GoodDescriptor object at ...>
print(MyClass2(42).attr)  # 42

## 8. Итоги и практические советы

### Что мы узнали?

- **Дескриптор** — объект с методами `__get__`, `__set__`, `__delete__`, который управляет доступом к атрибутам.
- **Data-дескрипторы** (`__set__` или `__delete__`) имеют приоритет над атрибутами экземпляра.
- **Non-data дескрипторы** (только `__get__`) можно переопределить атрибутом экземпляра.
- **`@property`** — это удобная обёртка над data-дескриптором.
- Дескрипторы позволяют создавать **переиспользуемую логику** управления атрибутами: валидацию, типизацию, ленивую загрузку, логирование.

### Когда использовать дескрипторы?

- Когда одна и та же логика доступа к атрибуту нужна во многих классах (например, валидация возраста, email, номера телефона).
- Когда нужно реализовать сложное поведение, которое неудобно делать через `@property` (например, дескриптор с параметрами).
- Когда вы пишете библиотеку или фреймворк и хотите предоставить элегантный API.

### Альтернативы дескрипторам:

- **`@property`** — для простых случаев (валидация одного атрибута в одном классе).
- **`__setattr__` / `__getattribute__`** — для глобального контроля над всеми атрибутами класса (но это сложнее и медленнее).
- **`__slots__`** — для экономии памяти (но не для контроля доступа).

### Ключевые моменты для запоминания

1. Дескриптор определяется **на уровне класса**, а не экземпляра.
2. В `__get__` всегда проверяйте `if obj is None:`.
3. Храните значения дескриптора в `obj.__dict__` под уникальным именем (обычно передаваемым в конструктор).
4. Data-дескрипторы **нельзя переопределить** атрибутом экземпляра — это часто полезно, но может быть неожиданным.
5. Дескрипторы — это мощный инструмент, но не злоупотребляйте им. Для простых случаев хватит `@property`.

